# ARYA RAG: Embedding Adapter

**Branch:** `shivam/dev/feature/rag-embedding-adapter`
**Location:** `backend/app/rag/` (root, because ingestion embeds chunks and retrieval embeds the query)

This notebook implements the `Embedder` port from the contract: text in, vectors out. It has **no database, SQL or schema code**. The only link to the schema is `dimension`, which must equal the pgvector column size (`vector(384)` for `all-MiniLM-L6-v2`).

**Contents**
1. `SentenceTransformerEmbedder`: real model, lazy loading, batching, normalised vectors, safe errors.
2. `FakeEmbedder`: deterministic and dependency-free, for tests (also usable by Person 3).
3. `assert_dimension`: startup check against the database column.
4. Tests that run without downloading a model.

**Rules**
- Importing the module must not load or download a model.
- Vectors are L2-normalised, which suits pgvector's cosine `<=>` operator.
- Failures during ingestion raise `IngestionError(EMBEDDING_FAILED)`. Failures while embedding a query raise `RetrievalError`. Messages never include the text being embedded.
- Every returned vector has exactly `dimension` floats, or the call fails.

**Model limit:** `all-MiniLM-L6-v2` truncates input at 256 tokens. Keep chunk size at or below that, as set in `ChunkingConfig`.

## 0. Install

In [1]:
%pip install -q sentence-transformers

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and contract fallbacks

In [2]:
from __future__ import annotations

import hashlib
import math
import re
import threading
from typing import Any, Callable, Protocol, runtime_checkable

# Use the real contract when running inside the repo, otherwise stand-ins.
try:
    from app.rag.contracts import IngestionError, IngestionErrorCode, RetrievalError
    from app.rag.ports import Embedder
except ImportError:
    from enum import Enum

    class IngestionErrorCode(str, Enum):
        EMBEDDING_FAILED = "EMBEDDING_FAILED"

    class IngestionError(Exception):
        def __init__(self, code, message):
            super().__init__(message)
            self.code, self.message = code, message

    class RetrievalError(Exception):
        pass

    @runtime_checkable
    class Embedder(Protocol):
        dimension: int
        def embed_texts(self, texts: list[str]) -> list[list[float]]: ...
        def embed_query(self, text: str) -> list[float]: ...

## 2. Vector helpers

In [3]:
def _to_float_list(vec: Any) -> list[float]:
    """Accept numpy arrays or plain sequences; return a list of Python floats."""
    if hasattr(vec, "tolist"):
        vec = vec.tolist()
    return [float(x) for x in vec]


def _normalise(vec: list[float]) -> list[float]:
    norm = math.sqrt(sum(x * x for x in vec))
    return [x / norm for x in vec] if norm > 0 else vec


def _batches(items: list[str], size: int):
    for i in range(0, len(items), size):
        yield items[i:i + size]

## 3. Real embedder (sentence-transformers)

- **Lazy loading:** the model loads on first use, guarded by a lock. Importing or constructing the class is cheap.
- **`model_factory`:** lets tests inject a stub model, so no download is needed.
- **Batching:** large documents are embedded in slices of `batch_size`.
- **Safe errors:** the underlying exception is chained (`from exc`) for logs, but the message shown to clients is generic.

In [4]:
DEFAULT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DEFAULT_DIMENSION = 384


class SentenceTransformerEmbedder:
    def __init__(self,
                 model_name: str = DEFAULT_MODEL,
                 dimension: int = DEFAULT_DIMENSION,
                 batch_size: int = 32,
                 model_factory: Callable[[str], Any] | None = None):
        if batch_size < 1:
            raise ValueError("batch_size must be >= 1")
        self.model_name = model_name
        self.dimension = dimension
        self.batch_size = batch_size
        self._factory = model_factory or self._default_factory
        self._model = None
        self._lock = threading.Lock()

    @staticmethod
    def _default_factory(name: str):
        from sentence_transformers import SentenceTransformer   # imported lazily
        return SentenceTransformer(name)

    def _get_model(self):
        if self._model is None:
            with self._lock:
                if self._model is None:
                    self._model = self._factory(self.model_name)
        return self._model

    def _encode(self, texts: list[str]) -> list[list[float]]:
        model = self._get_model()
        out: list[list[float]] = []
        for batch in _batches(texts, self.batch_size):
            raw = model.encode(batch, show_progress_bar=False)
            vectors = [_normalise(_to_float_list(v)) for v in raw]
            if len(vectors) != len(batch):
                raise ValueError("embedder returned the wrong number of vectors")
            for v in vectors:
                if len(v) != self.dimension:
                    raise ValueError(f"expected {self.dimension} dimensions, got {len(v)}")
            out.extend(vectors)
        return out

    def embed_texts(self, texts: list[str]) -> list[list[float]]:
        """Used during ingestion. Raises IngestionError(EMBEDDING_FAILED)."""
        if not texts:
            return []
        if any(not isinstance(t, str) or not t.strip() for t in texts):
            raise IngestionError(IngestionErrorCode.EMBEDDING_FAILED, "Cannot embed empty text.")
        try:
            return self._encode(texts)
        except Exception as exc:
            raise IngestionError(IngestionErrorCode.EMBEDDING_FAILED,
                                 "Embedding failed.") from exc

    def embed_query(self, text: str) -> list[float]:
        """Used during retrieval. Raises RetrievalError."""
        if not isinstance(text, str) or not text.strip():
            raise RetrievalError("Query is empty.")
        try:
            return self._encode([text])[0]
        except Exception as exc:
            raise RetrievalError("Could not embed the query.") from exc

## 4. Fake embedder (for tests)

A hashed bag-of-words: each word adds weight to a fixed vector position, then the vector is normalised. It is deterministic, needs no model, and texts that share words get a higher cosine similarity, so retrieval tests behave sensibly. Use a small `dimension` in tests to keep them fast.

In [5]:
_WORD_RE = re.compile(r"[a-z0-9]+")


class FakeEmbedder:
    def __init__(self, dimension: int = DEFAULT_DIMENSION):
        self.dimension = dimension

    def _vec(self, text: str) -> list[float]:
        v = [0.0] * self.dimension
        words = _WORD_RE.findall(text.lower())
        for w in words:
            idx = int(hashlib.sha256(w.encode()).hexdigest()[:8], 16) % self.dimension
            v[idx] += 1.0
        if not any(v):                                   # no words: fall back to a hash of the text
            idx = int(hashlib.sha256(text.encode()).hexdigest()[:8], 16) % self.dimension
            v[idx] = 1.0
        return _normalise(v)

    def embed_texts(self, texts: list[str]) -> list[list[float]]:
        return [self._vec(t) for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._vec(text)

## 5. Startup dimension check

Call this once at app startup, passing the dimension of the pgvector column. A mismatch then fails immediately instead of at the first insert.

In [6]:
def assert_dimension(embedder: Embedder, expected_db_dimension: int) -> None:
    if embedder.dimension != expected_db_dimension:
        raise RuntimeError(
            f"Embedder dimension {embedder.dimension} does not match the database "
            f"vector column ({expected_db_dimension}). Re-embed and migrate before changing models.")

## 6. Demo

In [7]:
fe = FakeEmbedder(dimension=64)
docs = ["Support vectors define the margin of an SVM",
        "Gradient descent minimises a loss function",
        "Kernels handle nonlinear classification"]
vecs = fe.embed_texts(docs)
q = fe.embed_query("what are support vectors")

def cos(a, b): return sum(x * y for x, y in zip(a, b))
for d, v in zip(docs, vecs):
    print(f"{cos(q, v):.3f}  {d}")

0.474  Support vectors define the margin of an SVM
0.000  Gradient descent minimises a loss function
0.000  Kernels handle nonlinear classification


## 7. Tests

The real model is replaced by a stub, so these run offline.

In [9]:
def expect(exc, fn):
    try:
        fn()
    except exc as e:
        return e
    raise AssertionError(f"expected {exc.__name__}")


class StubModel:
    """Mimics SentenceTransformer.encode and records calls."""
    def __init__(self, dim=8):
        self.dim, self.calls = dim, []
    def encode(self, batch, show_progress_bar=False):
        self.calls.append(list(batch))
        return [[float(len(t) % 5 + 1)] + [float(i) for i in range(self.dim - 1)] for t in batch]


class BrokenModel:
    def encode(self, batch, show_progress_bar=False):
        raise RuntimeError("secret internal detail: /tmp/model/path")


class WrongDimModel:
    def encode(self, batch, show_progress_bar=False):
        return [[1.0, 2.0] for _ in batch]


# --- port compliance
assert isinstance(FakeEmbedder(), Embedder)
assert isinstance(SentenceTransformerEmbedder(model_factory=lambda n: StubModel()), Embedder)

# --- lazy loading: constructing does not load the model
loads = []
emb = SentenceTransformerEmbedder(dimension=8, model_factory=lambda n: (loads.append(n), StubModel(8))[1])
assert loads == []
emb.embed_query("hello")
emb.embed_texts(["a b", "c d"])
assert loads == [DEFAULT_MODEL], "model should load once, on first use"

# --- batching
stub = StubModel(8)
emb = SentenceTransformerEmbedder(dimension=8, batch_size=3, model_factory=lambda n: stub)
out = emb.embed_texts([f"text {i}" for i in range(7)])
assert len(out) == 7
assert [len(c) for c in stub.calls] == [3, 3, 1]

# --- order is preserved and vectors are normalised
for v in out:
    assert len(v) == 8
    assert abs(math.sqrt(sum(x * x for x in v)) - 1.0) < 1e-9

# --- empty list is a no-op and never loads the model
loads.clear()
lazy = SentenceTransformerEmbedder(dimension=8, model_factory=lambda n: (loads.append(1), StubModel(8))[1])
assert lazy.embed_texts([]) == [] and loads == []

# --- blank inputs
e = expect(IngestionError, lambda: emb.embed_texts(["ok", "   "]))
assert e.code == IngestionErrorCode.EMBEDDING_FAILED
expect(RetrievalError, lambda: emb.embed_query("   "))

# --- errors are wrapped and do not leak internals
bad = SentenceTransformerEmbedder(dimension=8, model_factory=lambda n: BrokenModel())
e = expect(IngestionError, lambda: bad.embed_texts(["hello"]))
assert e.code == IngestionErrorCode.EMBEDDING_FAILED and "secret" not in e.message
assert isinstance(e.__cause__, RuntimeError)          # chained for logs only
e = expect(RetrievalError, lambda: bad.embed_query("hello"))
assert "secret" not in str(e)

# --- model load failure is also wrapped
def boom(name): raise OSError("download failed")
expect(IngestionError, lambda: SentenceTransformerEmbedder(model_factory=boom).embed_texts(["x"]))

# --- wrong dimension is rejected
wrong = SentenceTransformerEmbedder(dimension=8, model_factory=lambda n: WrongDimModel())
expect(IngestionError, lambda: wrong.embed_texts(["x"]))

# --- invalid config
expect(ValueError, lambda: SentenceTransformerEmbedder(batch_size=0))

# --- fake embedder: deterministic, normalised, right size
f = FakeEmbedder(dimension=64)
assert f.embed_query("support vectors") == f.embed_query("support vectors")
assert f.embed_texts(["a b c"]) == [f.embed_query("a b c")]
assert all(len(v) == 64 for v in f.embed_texts(["x", "y z", "!!!"]))
assert abs(math.sqrt(sum(x * x for x in f.embed_query("anything"))) - 1.0) < 1e-9

# --- fake embedder: similar text scores higher than unrelated text
fe = FakeEmbedder(dimension=384)
q = fe.embed_query("explain support vectors")
close = fe.embed_query("support vectors define the margin")
far = fe.embed_query("gradient descent minimises loss")
assert cos(q, close) > cos(q, far)

# --- dimension check
assert_dimension(FakeEmbedder(384), 384)
expect(RuntimeError, lambda: assert_dimension(FakeEmbedder(384), 768))

print("All embedding adapter tests passed")

All embedding adapter tests passed


## 8. Real model smoke test (optional)

Run this once on your machine. It downloads the model on first run, so it is not part of the automated tests.

In [10]:
try:
    real = SentenceTransformerEmbedder()
    v = real.embed_query("What is a support vector machine?")
    print("dimension:", len(v), "| normalised:", round(sum(x * x for x in v), 4))
    assert len(v) == real.dimension == 384
except Exception as e:
    print("Skipped (model not available):", type(e).__name__)

c:\Users\ShivamRajput\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2697.71it/s]


dimension: 384 | normalised: 1.0


## 9. Move this into the repository

| Notebook cells | Target file |
|---|---|
| 1 to 5 | `backend/app/rag/embedding.py` |
| 7 | `backend/tests/unit/rag/test_embedding.py`, rewritten as `pytest` functions |

`FakeEmbedder` can also move to `backend/tests/fakes/` so Person 3 can import it for agent tests.

**Before the PR**
1. Send Person 2 the model name and dimension (384) for the `vector(384)` column.
2. Run the smoke test (cell 8) once with the real model.
3. Run **Restart and Run All**, then **Clear All Outputs**, then commit.

**Not in this branch:** reading or writing chunks in a database, `ChunkRecord` assembly (the pgvector indexing branch does that), retrieval, API routes.